# UK Financial Data Pull
This notebook extracts open financial data relevant to wealth management with a primary focus on the UK market.

In [1]:
import pandas as pd
import requests
import os
import datetime
import re

OUTPUT_DIR = '../output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
def fetch_and_save_csv(url, filename, read_csv_kwargs=None):
    """
    Fetches a CSV from a URL and saves it. If the fetch fails, 
    it falls back to the existing local file (if available).
    read_csv_kwargs is passed to pd.read_csv to handle headers/skips.
    """
    if read_csv_kwargs is None: read_csv_kwargs = {}
    filepath = os.path.join(OUTPUT_DIR, filename)
    try:
        print(f"Fetching data from {url}...")
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status() # Raise an exception for bad status codes
        if '<!doctype html>' in response.text[:500].lower() or '<html' in response.text[:500].lower(): raise Exception('API returned HTML instead of CSV')
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(response.text)
        print(f"Successfully saved to {filepath}")
        
        # Validate it reads cleanly into pandas
        df = pd.read_csv(filepath, **read_csv_kwargs)
        return df
        
    except Exception as e:
        print(f"Warning: Failed to fetch data from {url}. Error: {e}")
        if os.path.exists(filepath):
            print(f"Falling back to existing local file: {filepath}")
            df = pd.read_csv(filepath, **read_csv_kwargs)
            return df
        else:
            print(f"Error: No local fallback exists for {filename}.")
            return None

## 1. Bank of England (BoE) Base Rate
The BoE publishes the official Bank Rate via their IADB programmatic interface.

In [3]:
# Fetching BoE Bank Rate History (IUDBEDR) since 1970
boe_url = 'https://www.bankofengland.co.uk/boeapps/database/_iadb-fromshowcolumns.asp?csv.x=yes&Datefrom=01/Jan/1970&Dateto=now&SeriesCodes=IUDBEDR&CSVF=TN&UsingCodes=Y&VPD=Y&VFD=N'
df_boe = fetch_and_save_csv(boe_url, 'boe_base_rate.csv')
if df_boe is not None:
    from IPython.display import display
    display(df_boe.head())


Fetching data from https://www.bankofengland.co.uk/boeapps/database/_iadb-fromshowcolumns.asp?csv.x=yes&Datefrom=01/Jan/1970&Dateto=now&SeriesCodes=IUDBEDR&CSVF=TN&UsingCodes=Y&VPD=Y&VFD=N...


Successfully saved to ../output\boe_base_rate.csv


,DATE,IUDBEDR
0,02 Jan 1975,11.5
1,03 Jan 1975,11.5
2,06 Jan 1975,11.5
3,07 Jan 1975,11.5
4,08 Jan 1975,11.5


## 1b. BoE Quoted Household Interest Rates
Expands the Bank of England data pipeline to natively pull average quoted Mortgage (2-year & 5-year fixed, 75% LTV) and Savings (Instant Access & 1-Year Bonds) interest rates.

In [ ]:
# Fetching BoE Quoted Household Rates since 2010 individually to bypass BoE multi-series restrictions
# Series Codes:
# IUMTLMV: 2yr Fixed Mortgage 75% LTV
# IUMBV34: 5yr Fixed Mortgage 75% LTV
# IUM55L2: 1yr Fixed Rate Bond
# IUM5957: Instant Access Deposit
import time
codes = ['IUMTLMV', 'IUMBV34', 'IUM55L2', 'IUM5957']
df_boe_rates = None
for code in codes:
    url = f"https://www.bankofengland.co.uk/boeapps/database/_iadb-fromshowcolumns.asp?csv.x=yes&Datefrom=01/Jan/2010&Dateto=now&SeriesCodes={code}&CSVF=TN&UsingCodes=Y&VPD=Y&VFD=N"
    df_temp = fetch_and_save_csv(url, f"boe_{code}.csv")
    if df_temp is not None:
        if df_boe_rates is None:
            df_boe_rates = df_temp
        else:
            df_boe_rates = pd.merge(df_boe_rates, df_temp, on='DATE', how='outer')
    time.sleep(1) # Be polite to BoE servers

if df_boe_rates is not None:
    try:
        df_boe_rates['DATE_DT'] = pd.to_datetime(df_boe_rates['DATE'])
        df_boe_rates = df_boe_rates.sort_values('DATE_DT').drop('DATE_DT', axis=1)
    except Exception as e:
        print(f"Error sorting dates: {e}")
    filepath = os.path.join(OUTPUT_DIR, 'boe_quoted_household_rates.csv')
    df_boe_rates.to_csv(filepath, index=False)
    print(f"Successfully consolidated BoE rates to {filepath}")
    from IPython.display import display
    display(df_boe_rates.tail())


## 2. HM Land Registry UK House Price Index
Monthly average house prices in the UK. This snippet checks recent months iteratively to dynamically grab the absolutely newest available dataset seamlessly.

In [4]:
def fetch_latest_hpi():
    base_url = "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/UK-HPI-full-file-{yyyy}-{mm:02d}.csv"
    date = datetime.date.today()
    year, month = date.year, date.month
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    for _ in range(12):
        url = base_url.format(yyyy=year, mm=month)
        print(f"Checking HPI URL: {url}...")
        try:
            response = requests.head(url, headers=headers, timeout=10)
            if response.status_code == 200:
                print(f"Found latest dataset at {url}")
                return url
        except Exception as e:
            print(f"Error checking {url}: {e}")
            
        month -= 1
        if month == 0:
            month = 12
            year -= 1
            
    print("Could not dynamically find a recent HPI dataset. Utilizing fallback...")
    return "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/UK-HPI-full-file-2023-12.csv" 

latest_hpi_url = fetch_latest_hpi()
if latest_hpi_url:
    df_hpi = fetch_and_save_csv(latest_hpi_url, 'uk_hpi_data.csv')
    if df_hpi is not None:
        from IPython.display import display
        display(df_hpi.tail())


Checking HPI URL: https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/UK-HPI-full-file-2026-03.csv...
Checking HPI URL: https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/UK-HPI-full-file-2026-02.csv...


Checking HPI URL: https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/UK-HPI-full-file-2026-01.csv...
Checking HPI URL: https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/UK-HPI-full-file-2025-12.csv...
Found latest dataset at https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/UK-HPI-full-file-2025-12.csv
Fetching data from https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/UK-HPI-full-file-2025-12.csv...


Successfully saved to ../output\uk_hpi_data.csv


,Date,RegionName,AreaCode,AveragePrice,Index,IndexSA,1m%Change,12m%Change,AveragePriceSA,SalesVolume,...,NewPrice,NewIndex,New1m%Change,New12m%Change,NewSalesVolume,OldPrice,OldIndex,Old1m%Change,Old12m%Change,OldSalesVolume
149080,01/08/2025,Yorkshire and The Humber,E12000003,206107,107.6,105.8,0.9,2.0,202741.0,5471.0,...,302627.0,112.1,-0.9,7.4,19.0,202629.0,107.3,1.0,1.8,5452.0
149081,01/09/2025,Yorkshire and The Humber,E12000003,206581,107.8,106.9,0.2,3.9,204762.0,4745.0,...,306535.0,113.5,1.3,10.2,18.0,202921.0,107.5,0.1,3.5,4727.0
149082,01/10/2025,Yorkshire and The Humber,E12000003,206401,107.7,106.7,-0.1,3.2,204537.0,4961.0,...,313072.0,115.9,2.1,10.3,8.0,202381.0,107.2,-0.3,2.8,4953.0
149083,01/11/2025,Yorkshire and The Humber,E12000003,209467,109.3,108.0,1.5,3.8,206882.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
149084,01/12/2025,Yorkshire and The Humber,E12000003,208447,108.8,108.0,-0.5,3.3,206964.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. ONS UK Inflation (CPI)
The Office for National Statistics publishes the authoritative Consumer Price Index (CPI All Items 2015=100) under time series 'D7G7'.

In [5]:
ons_cpi_url = 'https://www.ons.gov.uk/generator?format=csv&uri=/economy/inflationandpriceindices/timeseries/d7g7/mm23'
# The ONS generator exports heavily formatted CSVs with 8 metadata rows at the top.
df_ons_cpi = fetch_and_save_csv(ons_cpi_url, 'ons_cpi_inflation.csv', read_csv_kwargs={'skiprows': 8, 'names': ['Date', 'CPI_Rate']})
if df_ons_cpi is not None:
    from IPython.display import display
    display(df_ons_cpi.tail())


Fetching data from https://www.ons.gov.uk/generator?format=csv&uri=/economy/inflationandpriceindices/timeseries/d7g7/mm23...
Successfully saved to ../output\ons_cpi_inflation.csv


,Date,CPI_Rate
625,2025 SEP,3.8
626,2025 OCT,3.6
627,2025 NOV,3.2
628,2025 DEC,3.4
629,2026 JAN,3.0


## 4. UK Open Banking: Personal Current Accounts (PCA)
This aggregates live, real-time product data directly from Tier-1 UK banks using unauthenticated JSON endpoints mandated by the CMA9 Open Banking standard.

In [6]:
def fetch_open_banking_pca():
    banks = {
        'NatWest': 'https://openapi.natwest.com/open-banking/v2.2/personal-current-accounts',
        'Lloyds': 'https://api.lloydsbank.com/open-banking/v2.2/personal-current-accounts',
        'Barclays': 'https://atlas.api.barclays/open-banking/v2.2/personal-current-accounts'
    }
    
    all_products = []
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    for bank_name, url in banks.items():
        print(f"Fetching Open Banking Component from {bank_name}...")
        try:
            r = requests.get(url, headers=headers, timeout=10)
            if r.status_code == 200:
                data = r.json().get('data', [])
                for item in data:
                    for brand in item.get('Brand', []):
                        brand_name = brand.get('BrandName', bank_name)
                        for pca in brand.get('PCA', []):
                            segment = pca.get('Segment', [])
                            segment_str = ', '.join(segment) if isinstance(segment, list) else str(segment)
                            all_products.append({
                                'Bank': bank_name,
                                'Brand': brand_name,
                                'Product Name': pca.get('Name'),
                                'Segment': segment_str,
                            })
        except Exception as e:
            print(f"Warning: Failed to fetch from {bank_name}: {e}")
            
    if all_products:
        df = pd.DataFrame(all_products)
        filepath = os.path.join(OUTPUT_DIR, 'uk_open_banking_pca.csv')
        df.to_csv(filepath, index=False)
        print(f"Successfully saved aggregated Open Banking data to {filepath}")
        return df
    else:
        print("Warning: Failed to pull any Open Banking data dynamically.")
        fallback_path = os.path.join(OUTPUT_DIR, 'uk_open_banking_pca.csv')
        if os.path.exists(fallback_path):
            print(f"Falling back to existing local file: {fallback_path}")
            return pd.read_csv(fallback_path)
        return None

df_pca = fetch_open_banking_pca()
if df_pca is not None:
    from IPython.display import display
    display(df_pca.head())


Fetching Open Banking Component from NatWest...


Fetching Open Banking Component from Lloyds...


Fetching Open Banking Component from Barclays...
Successfully saved aggregated Open Banking data to ../output\uk_open_banking_pca.csv


,Bank,Brand,Product Name,Segment
0,NatWest,NatWest,Foundation,Basic
1,NatWest,NatWest,Select Account,General
2,NatWest,NatWest,Select Account with Overdraft Control,General
3,NatWest,NatWest,Select Private/Select Premier,General
4,NatWest,NatWest,Select Account (Credit Restricted),Basic


## 5. FCA Mortgage Lending Statistics (MLAR)
The Financial Conduct Authority publishes explicit mortgage lending administration data via complex multi-sheet Excel files. We elegantly extract the core "Summary 1" datasets down into native CSV.


In [7]:
fca_mlar_url = "https://www.fca.org.uk/publication/data/mlar-statistics-summary-long-run.xlsx"
filepath_xlsx = os.path.join(OUTPUT_DIR, 'fca_mlar_raw.xlsx')
filepath_csv = os.path.join(OUTPUT_DIR, 'fca_mlar_summary.csv')

try:
    print(f"Fetching FCA MLAR data from {fca_mlar_url}...")
    headers = {'User-Agent': 'Mozilla/5.0'}
    r = requests.get(fca_mlar_url, headers=headers, timeout=15)
    r.raise_for_status()
    
    with open(filepath_xlsx, 'wb') as f:
        f.write(r.content)
        
    # Convert the complex 'Summary 1' spreadsheet tab flawlessly into native CSV
    df_fca = pd.read_excel(filepath_xlsx, sheet_name='Summary 1', skiprows=5)
    df_fca.to_csv(filepath_csv, index=False)
    print(f"Successfully saved FCA MLAR to {filepath_csv}")
    
    from IPython.display import display
    display(df_fca.head())
except Exception as e:
    print(f"Warning: Failed to fetch FCA Data: {e}")
    if os.path.exists(filepath_csv):
        print(f"Falling back to existing local file: {filepath_csv}")
        df_fca = pd.read_csv(filepath_csv)
        from IPython.display import display
        display(df_fca.head())
    else:
        print("Error: No local fallback exists for FCA MLAR data.")


Fetching FCA MLAR data from https://www.fca.org.uk/publication/data/mlar-statistics-summary-long-run.xlsx...


Successfully saved FCA MLAR to ../output\fca_mlar_summary.csv


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 70,Unnamed: 71,Unnamed: 72,Unnamed: 73,Unnamed: 74,Unnamed: 75,Unnamed: 76,Unnamed: 77,Unnamed: 78,Unnamed: 79
0,Table (1),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Last updated: 10 March 2026
1,Residential loans to individuals(a),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Not seasonally adjusted,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Residential loans to individuals: Regulated a...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Companies House Corporate Demographics
This pulls the massive Free Company Data Product snapshot (~1GB+) dynamically without hardcoded URLs. To ensure pipeline performance and prevent memory overflow, it streams the ZIP directly from `download.companieshouse.gov.uk`, extracts a randomized contiguous 50,000 row sample representative of UK corporate health, and instantly securely discards the payload footprint.


In [8]:
def fetch_companies_house_sample():
    index_url = "http://download.companieshouse.gov.uk/en_output.html"
    print("Locating latest Companies House bulk data dump...")
    try:
        r = requests.get(index_url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)
        match = re.search(r'href="([^"]*BasicCompanyDataAsOneFile[^"]*\.zip)"', r.text)
        if not match:
            print("Could not find the BasicCompanyDataAsOneFile zip link on the index page.")
            raise Exception("Missing Regex Match on Frontpage")
            
        zip_url = "http://download.companieshouse.gov.uk/" + match.group(1)
        print(f"Streaming bulk data from: {zip_url} ... (This may take a moment)")
        
        zip_filepath = os.path.join(OUTPUT_DIR, 'ch_bulk.zip')
        with requests.get(zip_url, stream=True, headers={'User-Agent': 'Mozilla/5.0'}) as r:
            r.raise_for_status()
            with open(zip_filepath, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        
        print("Download complete. Extracting a 50,000 row sample to preserve operational memory limit...")
        
        # Extract sample using chunking directly from zip compression
        df_sample = pd.read_csv(zip_filepath, compression='zip', nrows=50000, low_memory=False)
        
        filepath = os.path.join(OUTPUT_DIR, 'companies_house_sample.csv')
        df_sample.to_csv(filepath, index=False)
        print(f"Successfully saved 50,000 row sample to {filepath}")
        
        # Clean up the massive 1GB zip file so the pipeline runs lean
        os.remove(zip_filepath)
        return df_sample
    except Exception as e:
        print(f"Failed to process Companies House data: {e}")
        fallback_path = os.path.join(OUTPUT_DIR, 'companies_house_sample.csv')
        if os.path.exists(fallback_path):
            print(f"Falling back to existing local file: {fallback_path}")
            return pd.read_csv(fallback_path, low_memory=False)
        return None

df_ch = fetch_companies_house_sample()
if df_ch is not None:
    from IPython.display import display
    display(df_ch.head())


Locating latest Companies House bulk data dump...
Streaming bulk data from: http://download.companieshouse.gov.uk/BasicCompanyDataAsOneFile-2026-03-02.zip ... (This may take a moment)


Download complete. Extracting a 50,000 row sample to preserve operational memory limit...


Successfully saved 50,000 row sample to ../output\companies_house_sample.csv


,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_7.CONDATE,PreviousName_7.CompanyName,PreviousName_8.CONDATE,PreviousName_8.CompanyName,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate
0,! LTD,08209948,NaN,NaN,9 PRINCES SQUARE,NaN,HARROGATE,NaN,ENGLAND,HG1 1ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25/09/2026,11/09/2025
1,!BIG IMPACT GRAPHICS LIMITED,11743365,NaN,NaN,124 CITY ROAD,NaN,LONDON,NaN,NaN,EC1V 2NX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29/12/2026,15/12/2025
2,!FE NETWORK LIMITED,16873705,NaN,NaN,C/O AACSL ACCOUNTANT LIMITED,1ST FLOOR NORTH WESTGATE HOUSE,"HARLOW, ESSEX",NaN,UNITED KINGDOM,CM20 1YS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,08/12/2026,NaN
3,!NFLECTION ADVISORY LIMITED,15073164,NaN,NaN,74 SANTERS LANE,NaN,POTTERS BAR,HERTFORDSHIRE,ENGLAND,EN6 2DA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28/08/2026,14/08/2025
4,!NFOGENIE LTD,13522064,NaN,NaN,71-75 SHELTON STREET,NaN,LONDON,GREATER LONDON,UNITED KINGDOM,WC2H 9JQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03/08/2026,20/07/2025


## 7. External Reference: FRED US Mortgage Rates
Used as an external macroeconomic reference. Utilizing pandas_datareader to natively interface with FRED without being blocked.

In [9]:
# FRED 30-Year Fixed Rate Mortgage Average in the US (MORTGAGE30US)
try:
    print("Fetching FRED US Mortgage Rates specifically via pandas_datareader...")
    import pandas_datareader.data as web
    start = datetime.datetime(2000, 1, 1)
    end = datetime.datetime.today()
    df_fred = web.DataReader('MORTGAGE30US', 'fred', start, end)
    filepath = os.path.join(OUTPUT_DIR, 'fred_30y_mortgage.csv')
    df_fred.to_csv(filepath)
    print(f"Successfully saved FRED data to {filepath}")
    from IPython.display import display
    display(df_fred.tail())
except Exception as e:
    print(f"Warning: Failed to fetch FRED Data: {e}")
    # Fallback to local file if available
    fallback_path = os.path.join(OUTPUT_DIR, 'fred_30y_mortgage.csv')
    if os.path.exists(fallback_path):
        print(f"Falling back to existing local file: {fallback_path}")
        df_fred = pd.read_csv(fallback_path)
        from IPython.display import display
        display(df_fred.tail())
    else:
        print("Error: No local fallback exists for fred_30y_mortgage.csv")


Fetching FRED US Mortgage Rates specifically via pandas_datareader...


Successfully saved FRED data to ../output\fred_30y_mortgage.csv


,MORTGAGE30US
DATE,
2026-02-19,6.01
2026-02-26,5.98
2026-03-05,6.00
2026-03-12,6.11
2026-03-19,6.22
